# Data Storytelling and Presenting Results

Notebook for **Chapter 5** of the *career* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/career/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Audience First: Headline, Evidence, Recommendation

## 5.2 Choosing the Right Chart and Removing Clutter

## 5.3 Executive Summaries and Handling Q&A

## 5.4 A Data Story Built Step by Step

## 5.5 Start from Zero: The Words and the Arithmetic Behind Every Story

Most failed presentations do not fail because of design. They fail because of arithmetic and vocabulary: someone says "up 10%" when they mean 10 points, or averages two averages, or compares a big region with a small one as if they were the same size. Before charts and slides, you need a small set of words and a small set of calculations that you never get wrong. This section builds them from nothing. Everything later in the chapter uses them.

In [ ]:
def pct_change(old, new):
    """Relative change in percent. Returns None when old is 0 (undefined)."""
    if old == 0:
        return None
    return (new - old) / old * 100

print(round(pct_change(80, 55), 1))
print(round(pct_change(55, 80), 1))
print(pct_change(0, 12))
print(round(pct_change(-20, -10), 1))

-31.2
45.5
None
-50.0


## 5.6 The Numbers Toolkit: Compute Before You Write

Section 5.1 gave you the storyline. This section gives you the calculations that feed it. There are five, and together they answer most business questions: *how much did it change* (Section 5.5), *where did the change come from* (contribution), *what is the right overall number* (weighted average), *how fast does it grow* (index and growth rate), and *is the overall picture hiding a different story inside the groups* (Simpson's paradox). The sixth idea, choosing between mean and median, finishes the section.

In [ ]:
import pandas as pd

sales = pd.DataFrame({"region": ["North", "West", "South"],
                      "Jan": [120, 95, 80],
                      "Mar": [130, 99, 55]}).set_index("region")
sales["change"] = sales["Mar"] - sales["Jan"]
sales["pct_change"] = (sales["change"] / sales["Jan"] * 100).round(1)
jan_total = sales["Jan"].sum()
sales["contribution_pts"] = (sales["change"] / jan_total * 100).round(1)
print(sales)
print("total change:", sales["change"].sum(), "lakh")
print("total pct change:", round(sales["change"].sum() / jan_total * 100, 1))
print("contributions add to:", round(sales["contribution_pts"].sum(), 1))

        Jan  Mar  change  pct_change  contribution_pts
region                                                
North   120  130      10         8.3               3.4
West     95   99       4         4.2               1.4
South    80   55     -25       -31.2              -8.5
total change: -11 lakh
total pct change: -3.7
contributions add to: -3.7


In [ ]:
import pandas as pd

r = pd.DataFrame({"region": ["A", "B", "C"],
                  "visits": [9000, 800, 200],
                  "conv_rate": [0.02, 0.10, 0.20]})
simple = r["conv_rate"].mean()
converted = r["visits"] * r["conv_rate"]
weighted = converted.sum() / r["visits"].sum()
print("converted per region:", converted.tolist())
print(f"simple average of rates: {simple:.2%}")
print(f"weighted (correct) overall: {weighted:.2%}")

converted per region: [180.0, 80.0, 40.0]
simple average of rates: 10.67%
weighted (correct) overall: 3.00%


In [ ]:
values = [100, 150, 75]            # yearly revenue: up 50%, then down 50%
index = [round(v / values[0] * 100) for v in values]
growth = [(b - a) / a * 100 for a, b in zip(values, values[1:])]
average_growth = sum(growth) / len(growth)
periods = len(values) - 1
cagr = ((values[-1] / values[0]) ** (1 / periods) - 1) * 100
print("index:", index)
print("yearly growth %:", growth)
print("average of yearly growth:", average_growth)
print("compound annual growth:", round(cagr, 1))

index: [100, 150, 75]
yearly growth %: [50.0, -50.0]
average of yearly growth: 0.0
compound annual growth: -13.4


In [ ]:
import pandas as pd

d = pd.DataFrame({"variant": ["A", "A", "B", "B"],
                  "segment": ["mobile", "desktop", "mobile", "desktop"],
                  "users":   [200, 800, 800, 200],
                  "converted": [16, 136, 72, 36]})
d["rate_pct"] = d["converted"] / d["users"] * 100
print(d)
total = d.groupby("variant")[["users", "converted"]].sum()
total["rate_pct"] = total["converted"] / total["users"] * 100
print(total)

  variant  segment  users  converted  rate_pct
0       A   mobile    200         16       8.0
1       A  desktop    800        136      17.0
2       B   mobile    800         72       9.0
3       B  desktop    200         36      18.0
         users  converted  rate_pct
variant                            
A         1000        152      15.2
B         1000        108      10.8


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
minutes = rng.lognormal(mean=1.0, sigma=0.8, size=1000)   # 1000 skewed ticket times
print(f"mean   {minutes.mean():.2f}")
print(f"median {np.median(minutes):.2f}")
print(f"p95    {np.percentile(minutes, 95):.2f}")
print(f"share slower than the mean: {(minutes > minutes.mean()).mean():.1%}")

mean   3.41
median 2.57
p95    9.11
share slower than the mean: 33.8%


## 5.7 Charts in Practice: Choose, Build and Fix

Section 5.2 gave rules. Here you apply them: you decide what a chart must show, build it in code, and repair bad ones. A chart is a claim. Your job is to make the claim obvious and impossible to misread.

In [ ]:
import matplotlib.pyplot as plt

months = ["Jan", "Feb", "Mar"]
series = {"North": [120, 125, 130], "West": [95, 97, 99], "South": [80, 60, 55]}
hero = "South"

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, values in series.items():
    is_hero = (name == hero)
    ax.plot(months, values, marker="o",
            color="#aa2d26" if is_hero else "#999999",
            linewidth=3 if is_hero else 1.5)
    ax.text(2.05, values[-1], f"{name} {values[-1]}", va="center",
            color="#aa2d26" if is_hero else "#666666",
            fontweight="bold" if is_hero else "normal")
ax.set_title("South sales fell 31% in two months", loc="left", fontweight="bold")
ax.set_ylabel("lakh rupees per month")
ax.set_ylim(0, 140)
ax.set_xlim(-0.2, 2.7)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
fig.savefig("south.png", dpi=110)
print("title words:", len(ax.get_title(loc="left").split()))
print("y-axis starts at:", ax.get_ylim()[0])
print("lines drawn:", len(ax.lines), "| legend present:", ax.get_legend() is not None)

title words: 7
y-axis starts at: 0.0
lines drawn: 3 | legend present: False


In [ ]:
def apparent_ratio(first, second, baseline):
    """How tall the second bar looks compared with the first."""
    return (second - baseline) / (first - baseline)

for baseline in (0, 200, 270, 280):
    ratio = apparent_ratio(295, 284, baseline)
    print(f"baseline {baseline:>3}: March looks {ratio:.0%} as tall as January")
print(f"true ratio: {284/295:.0%}")

baseline   0: March looks 96% as tall as January
baseline 200: March looks 88% as tall as January
baseline 270: March looks 56% as tall as January
baseline 280: March looks 27% as tall as January
true ratio: 96%


In [ ]:
stages = ["Visits", "Add to cart", "Start checkout", "Paid"]
counts = [10000, 3200, 900, 270]
for i in range(1, len(counts)):
    rate = counts[i] / counts[i - 1] * 100
    lost = counts[i - 1] - counts[i]
    print(f"{stages[i-1]} -> {stages[i]}: {rate:.1f}% continue, {lost:,} lost")
print(f"overall: {counts[-1] / counts[0] * 100:.1f}%")

Visits -> Add to cart: 32.0% continue, 6,800 lost
Add to cart -> Start checkout: 28.1% continue, 2,300 lost
Start checkout -> Paid: 30.0% continue, 630 lost
overall: 2.7%


## 5.8 Is the Change Real? Noise, Intervals and Honest Wording

Every real number has noise. If you toss a fair coin 100 times you will rarely see exactly 50 heads. If you send the same email to two groups of 10,000 people, their click rates will differ a little by pure luck. Before you say "B beats A", you must know how big the luck could be. This section gives you the plain-English version of the maths, the numbers to compute, and the sentences to say.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
a = rng.binomial(10000, 0.04, size=2000)   # 2000 repeats of "control": true rate 4%
b = rng.binomial(10000, 0.04, size=2000)   # 2000 repeats of "variant": the SAME true rate
diff_pts = (b - a) / 10000 * 100           # observed difference, in percentage points
print("typical size of the noise (std of diff):", round(diff_pts.std(), 2), "points")
print("smallest and largest difference:", round(diff_pts.min(), 2), round(diff_pts.max(), 2))
print("share of repeats with |diff| >= 0.5 points:", round(np.mean(np.abs(diff_pts) >= 0.5) * 100, 1), "%")

typical size of the noise (std of diff): 0.27 points
smallest and largest difference: -0.92 1.1
share of repeats with |diff| >= 0.5 points: 7.0 %


In [ ]:
import numpy as np
from scipy import stats

def compare_rates(conv_a, n_a, conv_b, n_b):
    p_a, p_b = conv_a / n_a, conv_b / n_b
    diff = p_b - p_a
    se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z = diff / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return diff, se, z, p_value, diff - 1.96 * se, diff + 1.96 * se

tests = {"Test 1": (400, 10000, 450, 10000),
         "Test 2": (80, 2000, 96, 2000),
         "Test 3": (4000, 100000, 4500, 100000)}
for name, args in tests.items():
    diff, se, z, p, lo, hi = compare_rates(*args)
    print(f"{name}: lift {diff*100:.2f} pts, SE {se*100:.3f}, z {z:.2f}, p {p:.4f}, interval {lo*100:+.2f} to {hi*100:+.2f} pts")

Test 1: lift 0.50 pts, SE 0.285, z 1.75, p 0.0796, interval -0.06 to +1.06 pts
Test 2: lift 0.80 pts, SE 0.648, z 1.23, p 0.2173, interval -0.47 to +2.07 pts
Test 3: lift 0.50 pts, SE 0.090, z 5.54, p 0.0000, interval +0.32 to +0.68 pts


In [ ]:
import math

def n_per_group(p, lift):
    """Users per group: 80% power, 5% significance, p = average baseline rate, lift in absolute terms."""
    return math.ceil(2 * (1.96 + 0.84) ** 2 * p * (1 - p) / lift ** 2)

print(n_per_group(0.0425, 0.005))
print(n_per_group(0.0425, 0.010))
print(n_per_group(0.05, 0.01))

25524
6381
7448


## 5.9 Summaries, Slides and Live Delivery

A correct analysis can still be ignored if it arrives in the wrong shape. Shape depends on who is reading and how much time they have. This section turns one result into three versions, a five-slide talk, an email, and a set of answers for hard questions. For the speaking technique itself see Chapter 3; for email craft see Chapter 4.

In [ ]:
import re

HEDGES = ["maybe", "possibly", "might", "seems", "somewhat", "perhaps", "kind of", "sort of", "it appears"]

def lint(summary):
    first_line = summary.strip().splitlines()[0]
    words = first_line.split()
    found = [h for h in HEDGES if h in summary.lower()]
    return {"headline_words": len(words),
            "headline_has_number": bool(re.search(r"\d", first_line)),
            "hedges": found}

draft = """It seems that sales might be somewhat lower this quarter.
Possibly one region is the reason, perhaps."""
better = """Sales fell 3.7% (295 to 284 lakh); South explains all of it.
South lost 25 lakh a month. Check data, stock and price this week."""
print("draft :", lint(draft))
print("better:", lint(better))

draft : {'headline_words': 10, 'headline_has_number': False, 'hedges': ['possibly', 'might', 'seems', 'somewhat', 'perhaps']}
better: {'headline_words': 12, 'headline_has_number': True, 'hedges': []}


## 5.10 How to Solve Any Data Storytelling and Presenting Results Problem

Storytelling problems look different on the surface (a table, a chart to fix, an A/B test, an angry director) but they follow the same loop. This section is the playbook: a decision flow, eight recurring patterns with a solved example each, a checklist for verifying an answer, and a debugging guide for when you are stuck.

In [ ]:
import numpy as np

# Pattern 2: where did it come from? (revenue by product, lakh)
old = {"Phones": 500, "Laptops": 300, "Accessories": 200}
new = {"Phones": 520, "Laptops": 240, "Accessories": 210}
old_total = sum(old.values())
for k in old:
    print(f"{k:12s} change {new[k]-old[k]:+4d}  contribution {(new[k]-old[k])/old_total*100:+.1f} pts")
print("total pct:", round((sum(new.values()) - old_total) / old_total * 100, 1))

# Pattern 3: compare groups fairly (channel, visits, conversion rate)
for name, n, p in [("Email", 20000, 0.03), ("Ads", 1000, 0.05), ("Search", 9000, 0.04)]:
    se = np.sqrt(p * (1 - p) / n)
    print(f"{name:7s} rate {p:.1%}  conversions {n*p:5.0f}  95% range {100*(p-1.96*se):.2f}% to {100*(p+1.96*se):.2f}%")

# Pattern 4: is it real? (3,000 users per arm, 150 vs 190 conversions)
a, b, n = 150, 190, 3000
pa, pb = a / n, b / n
se = np.sqrt(pa * (1 - pa) / n + pb * (1 - pb) / n)
d = pb - pa
print(f"lift {d*100:.2f} pts, range {100*(d-1.96*se):+.2f} to {100*(d+1.96*se):+.2f}, z {d/se:.2f}")

Phones       change  +20  contribution +2.0 pts
Laptops      change  -60  contribution -6.0 pts
Accessories  change  +10  contribution +1.0 pts
total pct: -3.0
Email   rate 3.0%  conversions   600  95% range 2.76% to 3.24%
Ads     rate 5.0%  conversions    50  95% range 3.65% to 6.35%
Search  rate 4.0%  conversions   360  95% range 3.60% to 4.40%
lift 1.33 pts, range +0.16 to +2.50, z 2.23


## 5.11 Going Deeper: Expert Notes, Edge Cases and Interview Probes

Everything above is enough to do the job well. This section is for when you want to be the person others ask. Read it after the drill if you are short of time.

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
sims, n, looks = 2000, 5000, [1000, 2000, 3000, 4000, 5000]
final_hits = any_hits = 0
for _ in range(sims):
    a = rng.random(n) < 0.04          # control: true rate 4%
    b = rng.random(n) < 0.04          # variant: the same true rate, no effect
    hit_at = []
    for k in looks:
        pa, pb = a[:k].mean(), b[:k].mean()
        se = np.sqrt(pa * (1 - pa) / k + pb * (1 - pb) / k)
        hit_at.append(abs(pb - pa) / se > 1.96)
    final_hits += hit_at[-1]
    any_hits += any(hit_at)
print("false wins, looking once at the end:", round(final_hits / sims * 100, 1), "%")
print("false wins, stopping at any of 5 looks:", round(any_hits / sims * 100, 1), "%")

false wins, looking once at the end: 5.6 %
false wins, stopping at any of 5 looks: 15.2 %


In [ ]:
for k in (1, 5, 20, 100):
    print(f"{k:>3} independent checks at the 5% level: chance of at least one false win = {1 - 0.95 ** k:.1%}")

  1 independent checks at the 5% level: chance of at least one false win = 5.0%
  5 independent checks at the 5% level: chance of at least one false win = 22.6%
 20 independent checks at the 5% level: chance of at least one false win = 64.2%
100 independent checks at the 5% level: chance of at least one false win = 99.4%


## 5.12 Mixed Practice Drill

These problems mix every skill in the chapter and do not tell you which pattern applies. Cover the answer, try it on paper, then check. If you miss one, the answer names the section to reread.

## 5.13 Self-Test and Where to Go Next

Answer all 15 questions without notes, then mark yourself. One point per correct answer.

## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/career/ch05.html